# Dense embeddings on Google Colab (Google Drive project)

This notebook runs [`scripts/build_dense_embeddings.py`](../scripts/build_dense_embeddings.py) from your **IR_project** folder on **Google Drive**.

**Important (GPU):** Do **not** run `%pip install -r requirements.txt` on Colab for this workflow. That file includes `torch>=...` from PyPI, which usually installs a **CPU-only** (`2.x+cpu`) wheel and replaces Colab’s CUDA-enabled PyTorch — then `Device: cpu` even with a GPU runtime. Use the cells in **§3** instead.

**Why Drive:** Colab mounts Drive at `/content/drive/MyDrive/...`. Set `IR_PROJECT_ROOT` so scripts find `data/raw_data/`.

**Requirements:** Runtime → **GPU** (T4 or better), synced project, and `documents.jsonl` at the path you pass to `--docs`.

**Outputs:** `models/doc_embeddings_*_{RUN_TAG}.npy`, `models/doc_ids_*_{RUN_TAG}.json`; Chroma under `chroma_db/` via **`--chroma`** (see **§8** — run MiniLM and bge-m3 **separately**, not `--model all` in one session). If Chroma fails with **HNSW / `InternalError`**, see **§9**.

## 1 · Mount Google Drive

Authorize when prompted. Your project path will look like `/content/drive/MyDrive/.../IR_project`.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## 2 · Set project root (edit this cell)

Set `PROJECT_ROOT` to the folder that contains `data/raw_data/` and `scripts/build_dense_embeddings.py`.

In [2]:
import os
from pathlib import Path

# --- EDIT: absolute path to IR_project on your mounted Drive ---
PROJECT_ROOT = Path(
    # "/content/drive/MyDrive/QMUL_temr_2/IR_project"
    "/content/drive/MyDrive/IR_project"
)

marker = PROJECT_ROOT / "data" / "raw_data"
if not marker.exists():
    raise FileNotFoundError(
        f"Cannot find {marker}. Update PROJECT_ROOT to your Drive path."
    )

os.environ["IR_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("cwd:", Path.cwd())
print("IR_PROJECT_ROOT:", os.environ["IR_PROJECT_ROOT"])

cwd: /content/drive/.shortcut-targets-by-id/1aXRy7gPN_tAjdkrH_t6YjkNZMgTm6Xc_/IR_project
IR_PROJECT_ROOT: /content/drive/MyDrive/IR_project


## 3 · GPU runtime + CUDA PyTorch (do this before other heavy `pip` installs)

1. **Runtime → Change runtime type → Hardware accelerator: GPU** (save, then continue).
2. Run the cells below. If `torch.cuda.is_available()` is `False` after the install step, Colab may still have a CPU-only `torch` — the uninstall + reinstall from `download.pytorch.org` fixes that.
3. CUDA 12.4 wheels (`cu124`) match many current Colab GPU images; if install fails, check [pytorch.org](https://pytorch.org/get-started/locally/) for the wheel that matches `nvidia-smi` / Colab’s CUDA.

In [3]:
# List GPUs (fails or empty if you are on CPU runtime)
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-f1802ca6-89c6-7d13-cc9c-1c1dd8ae5f73)


In [4]:
# Remove PyPI CPU-only torch if present, then install CUDA build from PyTorch
!pip uninstall -y torch torchvision torchaudio 2>/dev/null
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 116.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━

In [5]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (build):", getattr(torch.version, "cuda", None))
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("\nStill no GPU — confirm Runtime is GPU, rerun the cell above, or try cu121/cu118 wheels from pytorch.org.")

torch: 2.6.0+cu124
cuda available: True
cuda version (build): 12.4
device: Tesla T4


## 4 · Install dense-only dependencies

Uses [`requirements-colab-dense.txt`](../requirements-colab-dense.txt) in the repo — **no** `torch` line, so pip will not downgrade you to CPU-only.

In [6]:
%pip install -q -r requirements-colab-dense.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## 5 · (Optional) Hugging Face token

Public models (MiniLM, bge-m3) usually work without a token. For gated models: Runtime → Secrets → `HF_TOKEN`, or set `os.environ["HF_TOKEN"]`.

In [ ]:
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        os.environ["HF_TOKEN"] = tok.strip()
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("No HF_TOKEN in secrets (OK for public models).")
except Exception as e:
    print("Secrets not used:", e)

## 6 · Smoke test (`--limit`)

Confirm **`Device: cuda`** in the banner. Adjust `--docs` / `--limit` as needed.

In [ ]:
!python scripts/build_dense_embeddings.py \
  --docs data/json_format_data/subset_100k/documents.jsonl \
  --limit 500 \
  --model minilm \
  --run-tag COLAB_SMOKE

  Build dense embeddings
  Root         : /content/drive/MyDrive/QMUL_temr_2/IR_project
  Dataset      : data/json_format_data/subset_100k/documents.jsonl
  Models dir   : /content/drive/MyDrive/QMUL_temr_2/IR_project/models
  Chroma dir   : /content/drive/MyDrive/QMUL_temr_2/IR_project/chroma_db
  Limit        : 500
  RUN_TAG      : COLAB_SMOKE
  Models       : minilm
  Device       : cuda
  PyTorch      : 2.6.0+cu124
  Chroma       : no

Loaded 500 documents
  [MEM after corpus load] RSS=0.43 GB | Available=11.68 GB
  [minilm] matrix ~ 0.00 GB float32 (approx.)
------------------------------------------------------------
  minilm  dim=384  encode_batch=512  chroma_batch=5000
  npy: doc_embeddings_minilm_COLAB_SMOKE.npy
  collection: opensanctions_minilm_COLAB_SMOKE
------------------------------------------------------------
  [minilm] Cache hit — loading mmap doc_embeddings_minilm_COLAB_SMOKE.npy
  [MEM after minilm encode] RSS=0.78 GB | Available=11.47 GB

  Finished in 0.2 min
  R

## 7 · Full corpus: encode embeddings only (no Chroma)

These cells write `models/doc_embeddings_*_{RUN_TAG}.npy` and `doc_ids_*_{RUN_TAG}.json`. **`Chroma: no`** — population of Chroma is **§8**.

Use the **same `--run-tag`** for MiniLM and bge-m3 if you want matching filenames (e.g. `FULL_20260408`). One model per cell.

In [ ]:
# MiniLM — edit --docs / add --run-tag FULL_YYYYMMDD as needed
!python scripts/build_dense_embeddings.py \
  --docs data/json_format_data/full/documents.jsonl \
  --model minilm

  Build dense embeddings
  Root         : /content/drive/MyDrive/QMUL_temr_2/IR_project
  Dataset      : data/json_format_data/full/documents.jsonl
  Models dir   : /content/drive/MyDrive/QMUL_temr_2/IR_project/models
  Chroma dir   : /content/drive/MyDrive/QMUL_temr_2/IR_project/chroma_db
  Limit        : ALL
  RUN_TAG      : FULL_20260408
  Models       : minilm
  Device       : cuda
  PyTorch      : 2.6.0+cu124
  Chroma       : no

Loaded 1,245,931 documents
  [MEM after corpus load] RSS=5.93 GB | Available=6.20 GB
  [minilm] matrix ~ 1.91 GB float32 (approx.)
------------------------------------------------------------
  minilm  dim=384  encode_batch=512  chroma_batch=5000
  npy: doc_embeddings_minilm_FULL_20260408.npy
  collection: opensanctions_minilm_FULL_20260408
------------------------------------------------------------
  [minilm] Loading 'all-MiniLM-L6-v2' on CUDA …
modules.json: 100% 349/349 [00:00<00:00, 1.77MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:

In [ ]:
# bge-m3 (wrapper; same as --model bge-m3)
!python scripts/encode_dense_bge_m3.py \
  --docs data/json_format_data/full/documents.jsonl

  Build dense embeddings
  Root         : /content/drive/MyDrive/QMUL_temr_2/IR_project
  Dataset      : data/json_format_data/full/documents.jsonl
  Models dir   : /content/drive/MyDrive/QMUL_temr_2/IR_project/models
  Chroma dir   : /content/drive/MyDrive/QMUL_temr_2/IR_project/chroma_db
  Limit        : ALL
  RUN_TAG      : FULL_20260408
  Models       : bge-m3
  Device       : cuda
  PyTorch      : 2.6.0+cu124
  Chroma       : no

Loaded 1,245,931 documents
  [MEM after corpus load] RSS=5.92 GB | Available=6.21 GB
  [bge-m3] matrix ~ 5.10 GB float32 (approx.)
------------------------------------------------------------
  bge-m3  dim=1024  encode_batch=128  chroma_batch=5000
  npy: doc_embeddings_bge_m3_FULL_20260408.npy
  collection: opensanctions_bge_m3_FULL_20260408
------------------------------------------------------------
  [bge-m3] Loading 'BAAI/bge-m3' on CUDA …
modules.json: 100% 349/349 [00:00<00:00, 1.72MB/s]
config_sentence_transformers.json: 100% 123/123 [00:00<00:00, 

## 8 · Chroma: run **one model per session** (recommended)

A single `--model all --chroma` run can exhaust RAM or disconnect during the **second** Chroma insert. Prefer:

1. Complete **§7** so both `.npy` caches exist for your `RUN_TAG`.
2. Run **MiniLM + Chroma** below (loads cached `.npy`, inserts one collection).
3. Optionally **Runtime → Restart runtime** to free RAM, then run **bge-m3 + Chroma**.

Both write under the same `chroma_db/` path but **different** collections (`opensanctions_minilm_<RUN_TAG>` vs `opensanctions_bge_m3_<RUN_TAG>`).

**Edit** `--run-tag` to match §7 (default below: `FULL_20260408`).

In [7]:
# MiniLM + Chroma (after §7 MiniLM .npy exists; reuses cache)
!python scripts/build_dense_embeddings.py \
  --docs data/json_format_data/full/documents.jsonl \
  --model minilm \
  --chroma \
  --run-tag FULL_20260408

  Build dense embeddings
  Root         : /content/drive/MyDrive/IR_project
  Dataset      : data/json_format_data/full/documents.jsonl
  Models dir   : /content/drive/MyDrive/IR_project/models
  Chroma dir   : /content/drive/MyDrive/IR_project/chroma_db
  Limit        : ALL
  RUN_TAG      : FULL_20260408
  Models       : minilm
  Device       : cuda
  PyTorch      : 2.6.0+cu124
  Chroma       : yes

Loaded 1,245,931 documents
  [MEM after corpus load] RSS=5.93 GB | Available=6.12 GB
  [minilm] matrix ~ 1.91 GB float32 (approx.)
------------------------------------------------------------
  minilm  dim=384  encode_batch=512  chroma_batch=5000
  npy: doc_embeddings_minilm_FULL_20260408.npy
  collection: opensanctions_minilm_FULL_20260408
------------------------------------------------------------
  [minilm] Cache hit — loading mmap doc_embeddings_minilm_FULL_20260408.npy
  [MEM after minilm encode] RSS=6.29 GB | Available=5.90 GB
  [minilm] Inserting 1,245,931 docs (chroma_batch=5000)…

### bge-m3 + Chroma

Run **after** the MiniLM + Chroma cell finishes. If Colab feels tight on RAM, use **Runtime → Restart runtime**, remount Drive, `cd` to `PROJECT_ROOT`, reinstall only **`requirements-colab-dense.txt`** (skip full torch reinstall if CUDA still works), then run this cell.

In [8]:
# bge-m3 + Chroma (reuses cached doc_embeddings_bge_m3_<RUN_TAG>.npy from §7)
!python scripts/build_dense_embeddings.py \
  --docs data/json_format_data/full/documents.jsonl \
  --model bge-m3 \
  --chroma \
  --run-tag FULL_20260408

  Build dense embeddings
  Root         : /content/drive/MyDrive/IR_project
  Dataset      : data/json_format_data/full/documents.jsonl
  Models dir   : /content/drive/MyDrive/IR_project/models
  Chroma dir   : /content/drive/MyDrive/IR_project/chroma_db
  Limit        : ALL
  RUN_TAG      : FULL_20260408
  Models       : bge-m3
  Device       : cuda
  PyTorch      : 2.6.0+cu124
  Chroma       : yes

Loaded 1,245,931 documents
  [MEM after corpus load] RSS=5.92 GB | Available=6.12 GB
  [bge-m3] matrix ~ 5.10 GB float32 (approx.)
------------------------------------------------------------
  bge-m3  dim=1024  encode_batch=128  chroma_batch=5000
  npy: doc_embeddings_bge_m3_FULL_20260408.npy
  collection: opensanctions_bge_m3_FULL_20260408
------------------------------------------------------------
  [bge-m3] Cache hit — loading mmap doc_embeddings_bge_m3_FULL_20260408.npy
  [MEM after bge-m3 encode] RSS=6.28 GB | Available=5.87 GB
  [bge-m3] Inserting 1,245,931 docs (chroma_batch=5000)

### Advanced (not recommended)

Single session: `python scripts/build_dense_embeddings.py --docs ... --model all --chroma` — can work but often **OOMs or disconnects** on the second Chroma insert. Prefer §8 as written.

## 9 · Troubleshooting: Chroma `InternalError` / HNSW segment reader

If the traceback mentions **`Error constructing hnsw segment reader`** or fails on **`coll.count()`**, the **on-disk Chroma database** under `chroma_db/` is **corrupted or incomplete**. Typical causes:

- A **previous run stopped** mid-insert (disconnect, crash, kernel stop).
- **`chroma_db` on Google Drive** — network filesystem sync can break SQLite / HNSW segments (not as reliable as local disk).

**What is *not* broken:** your **`doc_embeddings_*.npy`** in `models/` — they are independent. You usually **do not** need to re-encode.

### Recovery

**A. Rebuild the collection with `--force-chroma`** (script deletes that collection and re-upserts when the collection was already “full”):

```bash
python scripts/build_dense_embeddings.py \
  --docs data/json_format_data/full/documents.jsonl \
  --model bge-m3 \
  --chroma \
  --run-tag FULL_20260408 \
  --force-chroma
```

If **`count()` crashes before** the script can delete the collection, **B** below.

**B. Delete bad Chroma data, then re-run §8**

- **Nuclear:** remove the whole `chroma_db` folder on your project path, then run the MiniLM + Chroma cell and the bge-m3 + Chroma cell again (`.npy` files stay; only Chroma is rebuilt).
- **Or** delete only the broken collection via a small Python snippet (see next cell).

**C. Safer for huge jobs:** write Chroma to **Colab local disk** first, then copy to Drive:

```bash
python scripts/build_dense_embeddings.py \
  --docs data/json_format_data/full/documents.jsonl \
  --model bge-m3 \
  --chroma \
  --run-tag FULL_20260408 \
  --chroma-dir /content/chroma_work
```

After success, zip `/content/chroma_work` and upload to your project, or sync — avoid heavy concurrent Drive sync during insert.

**Note:** `--reset` on the script also **deletes** matching `.npy` / `doc_ids` for that run tag — use it only if you really want a full re-encode, not for “Chroma only” fixes.

In [ ]:
# Optional: delete one broken collection by name (if opening the client works).
# If this also errors, delete the whole `chroma_db` folder in Drive / file browser.
import os
from pathlib import Path

import chromadb

root = Path(os.environ.get("IR_PROJECT_ROOT", "."))
chroma_dir = root / "chroma_db"
COLLECTION = "opensanctions_bge_m3_FULL_20260408"  # edit to match your RUN_TAG

client = chromadb.PersistentClient(path=str(chroma_dir))
try:
    client.delete_collection(COLLECTION)
    print("Deleted collection:", COLLECTION)
except Exception as e:
    print("Could not delete (corrupt DB or wrong name):", e)
    print("Fallback: remove folder", chroma_dir, "in Drive, then re-run §8.")